In [8]:
from sklearn.model_selection import train_test_split
import pandas as pd

train = pd.read_csv('../data/train_processed.csv')

In [9]:
#데이터 분리
X = train.drop('credit', axis=1)
y = train['credit']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
cat_features = X.select_dtypes(include='object').columns.tolist()
cat_features


C:\Users\ghkdr\AppData\Local\Temp\ipykernel_34992\451508059.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include='object').columns.tolist()


['gender',
 'car',
 'reality',
 'income_type',
 'edu_type',
 'family_type',
 'house_type',
 'occyp_type']

In [15]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features,
    early_stopping_rounds=100
)


0:	learn: 1.0698706	test: 1.0697100	best: 1.0697100 (0)	total: 158ms	remaining: 2m 38s
100:	learn: 0.8017817	test: 0.8050119	best: 0.8050119 (100)	total: 4.1s	remaining: 36.5s
200:	learn: 0.7902028	test: 0.8004204	best: 0.8004204 (200)	total: 8.16s	remaining: 32.5s
300:	learn: 0.7737390	test: 0.7949465	best: 0.7949465 (300)	total: 12.5s	remaining: 29.1s
400:	learn: 0.7599366	test: 0.7911313	best: 0.7911313 (400)	total: 17s	remaining: 25.4s
500:	learn: 0.7483591	test: 0.7886016	best: 0.7885908 (499)	total: 21.4s	remaining: 21.3s
600:	learn: 0.7371544	test: 0.7858766	best: 0.7858391 (596)	total: 25.6s	remaining: 17s
700:	learn: 0.7263680	test: 0.7837020	best: 0.7836782 (699)	total: 29.9s	remaining: 12.8s
800:	learn: 0.7149075	test: 0.7810816	best: 0.7810816 (800)	total: 34.2s	remaining: 8.49s
900:	learn: 0.7043584	test: 0.7787340	best: 0.7787340 (900)	total: 38.4s	remaining: 4.22s
999:	learn: 0.6936384	test: 0.7769627	best: 0.7769627 (999)	total: 42.8s	remaining: 0us

bestTest = 0.776962

In [16]:
from sklearn.metrics import accuracy_score, f1_score

pred = model.predict(X_valid)

acc = accuracy_score(y_valid, pred)
f1 = f1_score(y_valid, pred, average='macro')

print(f"Accuracy: {acc:.4f}")
print(f"F1 Macro: {f1:.4f}")


Accuracy: 0.6967
F1 Macro: 0.4061


In [17]:
import pandas as pd

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance()
}).sort_values(by='importance', ascending=False)

feature_importance.head(10)


,feature,importance
15,begin_month,24.658536
16,age,10.944904
13,occyp_type,9.338169
4,income_total,8.688016
18,income_per_person,7.539994
17,employment_years,7.209177
7,family_type,5.766943
6,edu_type,5.202519
5,income_type,5.168425
8,house_type,2.523217
